# Глава 1. Анализ датасета USA Banking Transactions Dataset (2023-2024)

**Целевой признак:** `Transaction_Type` (Debit / Credit)  
**Объём:** 5000 записей, 16 признаков  
**Лицензия:** Открытый доступ (Kaggle)  
**Дата создания:** 20.01.2025  
**Источник:** https://www.kaggle.com/datasets/pradeepkumar2424/usa-banking-transactions-dataset-2023-2024

In [1]:
# Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.preprocessing import LabelEncoder

# Настройка стилей
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_style("whitegrid")
sns.set_palette("husl")
np.random.seed(42)

In [2]:
# Загрузка данных (положите CSV в ту же папку или укажите полный путь)
df = pd.read_csv('Banking_Transactions_USA_2023_2024.csv')
print(f"Размер датасета: {df.shape}")
print("\nИнформация о признаках:")
print(df.info())
print("\nПервые 3 строки:")
display(df.head(3))

FileNotFoundError: [Errno 2] No such file or directory: 'Banking_Transactions_USA_2023_2024.csv'

In [ ]:
# 2.1 Гистограммы числовых признаков
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

axes[0].hist(df['Transaction_Amount'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('Распределение суммы транзакций', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Сумма (USD)')

axes[1].hist(df['Customer_Age'], bins=20, edgecolor='black', alpha=0.7, color='seagreen')
axes[1].set_title('Распределение возраста клиентов', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Возраст (лет)')

axes[2].hist(df['Customer_Income'], bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[2].set_title('Распределение годового дохода', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Доход (USD)')

axes[3].hist(df['Account_Balance'], bins=30, edgecolor='black', alpha=0.7, color='purple')
axes[3].set_title('Распределение баланса счёта', fontsize=12, fontweight='bold')
axes[3].set_xlabel('Баланс (USD)')

axes[4].hist(df['Loyalty_Points_Earned'], bins=25, edgecolor='black', alpha=0.7, color='goldenrod')
axes[4].set_title('Распределение бонусных баллов', fontsize=12, fontweight='bold')
axes[4].set_xlabel('Баллы')

fig.delaxes(axes[5])
plt.tight_layout()
plt.show()

In [ ]:
# 2.2 Визуализация seaborn
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='Transaction_Type', y='Transaction_Amount', hue='Transaction_Type', legend=False)
plt.title('Сумма транзакции по типам операций', fontsize=14, fontweight='bold')
plt.show()

plt.figure(figsize=(12, 6))
sns.violinplot(data=df, x='Transaction_Type', y='Customer_Age', hue='Transaction_Type', legend=False)
plt.title('Возраст клиентов по типам транзакций', fontsize=14, fontweight='bold')
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Transaction_Amount', y='Account_Balance', hue='Transaction_Type', alpha=0.7, s=80)
plt.title('Зависимость баланса от суммы транзакции', fontsize=14, fontweight='bold')
plt.legend(title='Тип транзакции', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 2.3 Интерактивные графики Plotly
fig1 = px.histogram(df, x='Category', y='Transaction_Amount', color='Transaction_Type',
                    title='Распределение сумм транзакций по категориям', barmode='group')
fig1.show()

fig2 = px.scatter_3d(df, x='Transaction_Amount', y='Customer_Age', z='Account_Balance',
                     color='Transaction_Type', size='Customer_Income',
                     title='3D: сумма, возраст, баланс (размер = доход)')
fig2.show()

In [ ]:
# 2.5 Тепловые карты
numeric_cols = ['Transaction_Amount', 'Customer_Age', 'Customer_Income', 'Account_Balance', 'Loyalty_Points_Earned']
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, square=True, linewidths=1, fmt='.2f', cbar_kws={'shrink': 0.8})
plt.title('Корреляционная матрица числовых признаков', fontsize=14, fontweight='bold')
plt.show()

pivot_table = df.pivot_table(values='Transaction_Amount', index='Category', columns='Payment_Method', aggfunc='mean', fill_value=0)
plt.figure(figsize=(14, 8))
sns.heatmap(pivot_table, annot=True, cmap='YlOrRd', fmt='.0f', cbar_kws={'label': 'Средняя сумма (USD)'})
plt.title('Средняя сумма транзакции по категории и способу платежа', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 2.6 Дубликаты и 2.7 Выбросы (IQR)
print(f"Дубликатов: {df.duplicated().sum()}")
print(f"Дубликатов по Transaction_ID: {df.duplicated(subset=['Transaction_ID']).sum()}")

def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    return data[(data[column] < lower) | (data[column] > upper)], lower, upper

out_amt, lb_a, ub_a = detect_outliers_iqr(df, 'Transaction_Amount')
out_inc, lb_i, ub_i = detect_outliers_iqr(df, 'Customer_Income')

print(f"Выбросы Amount: {len(out_amt)} ({len(out_amt)/len(df)*100:.2f}%)")
print(f"Выбросы Income: {len(out_inc)} ({len(out_inc)/len(df)*100:.2f}%)")

In [ ]:
# 2.8 Условная фильтрация
filter1 = df[(df['Fraud_Flag'] == 1) & (df['Transaction_Amount'] > 1000)]
filter2 = df[(df['Customer_Income'] > 150000) & (df['Category'] == 'Electronics')]
filter3 = df[(df['Transaction_Status'] == 'Success') & (df['Discount_Applied'] == True) & (df['Category'] == 'Food')]

print(f"Фильтр 1 (Fraud > 1000$): {len(filter1)}")
print(f"Фильтр 2 (Income > 150k & Electronics): {len(filter2)}")
print(f"Фильтр 3 (Success & Discount & Food): {len(filter3)}")

In [ ]:
# 2.9 Добавление шума
def add_gaussian_noise(data, column, noise_level=0.01):
    noise = np.random.normal(0, noise_level * data[column].std(), len(data))
    return (data[column] + noise).clip(lower=0)

def add_poisson_noise(data, column, noise_scale=0.05):
    noise = np.random.poisson(noise_scale * data[column].mean(), len(data))
    return (data[column] + noise).clip(lower=0)

df_noisy = df.copy()
df_noisy['Transaction_Amount_noisy'] = add_gaussian_noise(df, 'Transaction_Amount', 0.01)
df_noisy['Loyalty_Points_Earned_noisy'] = add_poisson_noise(df, 'Loyalty_Points_Earned', 0.05)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['Transaction_Amount'], bins=30, alpha=0.5, label='Original', color='blue', edgecolor='black')
axes[0].hist(df_noisy['Transaction_Amount_noisy'], bins=30, alpha=0.5, label='С шумом', color='red', edgecolor='black')
axes[0].legend()

axes[1].hist(df['Loyalty_Points_Earned'], bins=30, alpha=0.5, label='Original', color='blue', edgecolor='black')
axes[1].hist(df_noisy['Loyalty_Points_Earned_noisy'], bins=30, alpha=0.5, label='С шумом', color='red', edgecolor='black')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2.10 Биннинг числовых признаков
amount_bins = [0, 50, 200, 500, 1000, 5000, float('inf')]
amount_labels = ['Micro (<50)', 'Small (50-200)', 'Medium (200-500)', 'Large (500-1000)', 'Very Large (1k-5k)', 'Huge (>5k)']
df['Amount_Category'] = pd.cut(df['Transaction_Amount'], bins=amount_bins, labels=amount_labels, right=False)

age_bins = [18, 25, 35, 45, 55, 70]
age_labels = ['18-25', '26-35', '36-45', '46-55', '56-70']
df['Age_Group'] = pd.cut(df['Customer_Age'], bins=age_bins, labels=age_labels, right=False)

income_bins = [0, 30000, 60000, 100000, 150000, float('inf')]
income_labels = ['Low (<30k)', 'Lower-Mid (30k-60k)', 'Mid (60k-100k)', 'Upper-Mid (100k-150k)', 'High (>150k)']
df['Income_Group'] = pd.cut(df['Customer_Income'], bins=income_bins, labels=income_labels, right=False)

print(df['Amount_Category'].value_counts())
print(df['Income_Group'].value_counts())

In [ ]:
# 2.11 Дополнительные преобразования
transaction_mapping = {'Debit': 'Списание', 'Credit': 'Зачисление'}
df['Transaction_Type_RU'] = df['Transaction_Type'].map(transaction_mapping)
df['Transaction_Status'] = df['Transaction_Status'].str.capitalize()

df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])
df['Transaction_Year'] = df['Transaction_Date'].dt.year
df['Transaction_Month'] = df['Transaction_Date'].dt.month
df['Transaction_Hour'] = df['Transaction_Date'].dt.hour

df['Amount_to_Income_Ratio'] = df['Transaction_Amount'] / (df['Customer_Income'] / 12)

fraud_ratio = df[df['Fraud_Flag'] == 1]['Amount_to_Income_Ratio'].mean()
legit_ratio = df[df['Fraud_Flag'] == 0]['Amount_to_Income_Ratio'].mean()
print(f"Среднее ratio (Fraud): {fraud_ratio:.4f}")
print(f"Среднее ratio (Legit): {legit_ratio:.4f}")

In [ ]:
# 3. Категориальные признаки и кодирование
print("=== Категориальные признаки ===")
for col in ['Transaction_Type', 'Customer_Gender', 'Category', 'Payment_Method', 'Transaction_Status', 'Discount_Applied']:
    print(f"{col}: {df[col].nunique()} уникальных значений")

# Label Encoding
le_gender = LabelEncoder()
df['Gender_Encoded'] = le_gender.fit_transform(df['Customer_Gender'])

le_status = LabelEncoder()
df['Status_Encoded'] = le_status.fit_transform(df['Transaction_Status'])

# One-Hot Encoding
ohe_payment = pd.get_dummies(df['Payment_Method'], prefix='payment')
df = pd.concat([df, ohe_payment], axis=1)

# Frequency Encoding
category_freq = df['Category'].value_counts(normalize=True)
df['Category_Freq_Encoded'] = df['Category'].map(category_freq)

print("\nКодирование завершено. Новые столбцы добавлены.")

## 📌 Выводы и гипотезы

1. **Синтетическая природа** данных позволяет безопасно тестировать модели, но требует валидации на реальных выборках.
2. **Гипотеза 1:** Доход положительно коррелирует со средней суммой транзакций в категориях `Electronics` и `Entertainment`.
3. **Гипотеза 2:** Мошеннические операции чаще происходят в ночные часы и выходные дни.
4. **Этические риски:** Возможна географическая или гендерная дискриминация при автоматическом скоринге.
5. **Перспективы расширения:** Добавление `device_type`, `ip_geolocation`, `transaction_history_windows` позволит решать задачи real-time fraud detection и персонализации.

📚 **Источник:** Pradeep Kumar. USA Banking Transactions Dataset (2023-2024). Kaggle.